# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library, following the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema JSON-LD file URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Get and print dataset metadata using .to_json()
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

# Optionally, print version and published date
print(f"Version: {metadata.get('version', 'N/A')}")
print(f"Published date: {metadata.get('datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets, their fields, and IDs.

The FAIR^2 dataset may contain one or more record sets: below, we identify their `@id`s, list their fields (columns), and show the corresponding IDs.


In [ ]:
# List all record set @ids and their field (column) @ids, with labels
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  name: {getattr(rs, 'name', '')}")
    if hasattr(rs, 'description'):
        print(f"  description: {rs.description}")
    print("  Fields/Columns:")
    for field in rs.fields:
        print(f"    - @id: {field.id:60s} label: {getattr(field, 'name', '')}")
    print("\n-------------------\n")

## 3. Data Extraction
Load data from one or more record sets into a DataFrame for analysis.

> **Tip:** Use the `@id` values listed above. Update the variable `record_set_ids` below to contain the `@id` of each record set you wish to load.

In [ ]:
# List of available record set IDs discovered above
record_sets = dataset.metadata.record_sets
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    # Load all records for this record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for RecordSet {record_set_id}. Columns:")
        print(df.columns.tolist())
        print(df.head(2), '\n')
    else:
        print(f"No records found for RecordSet {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps to one of the main data tables.
Let's select the main protein abundance record set (choose the @id with the most feature columns/data) and operate on numeric columns.

In [ ]:
# Choose a main record set: pick the first with data and numeric columns
main_rs_id = None
numeric_fields = []
for rs in dataset.metadata.record_sets:
    df = dataframes.get(rs.id, pd.DataFrame())
    if not df.empty:
        # Try to find numeric columns via field dataType
        for field in rs.fields:
            if getattr(field, 'data_type', '').lower() in ['float', 'integer', 'number']:
                numeric_fields.append(field.id)
        if numeric_fields:
            main_rs_id = rs.id
            break

if not main_rs_id or not numeric_fields:
    print("Could not locate numeric fields in available record sets.")
else:
    df = dataframes[main_rs_id]
    print(f"Main RecordSet ID: {main_rs_id}")
    print(f"Numeric Field Candidates: {numeric_fields}")

    # For demonstration, pick the first numeric field
    numeric_field = numeric_fields[0]
    print(f"Using numeric field: {numeric_field}")

    # Remove outliers, filter, normalize, group, etc.
    # Choose a threshold (e.g., mean of the field)
    if pd.api.types.is_numeric_dtype(df[numeric_field]):
        threshold = df[numeric_field].mean()
    else:
        # Try convert
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > mean ({threshold:.3f}): {len(filtered_df)} rows")
    print(filtered_df[[numeric_field]].head())

    # Normalize field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by any categorical field (choose first string-type column)
    group_field = None
    for col in df.columns:
        if col == numeric_field: continue
        if df[col].dtype == object:
            group_field = col
            break
    if group_field:
        print(f"Grouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset for the selected record set/columns.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and numeric_fields:
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # If we have a group field
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(12, 6))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

    # Scatter plot if >1 numeric field
    if len(numeric_fields) > 1:
        numeric_field2 = numeric_fields[1]
        plt.figure(figsize=(7,5))
        plt.scatter(df[numeric_field], df[numeric_field2], alpha=0.5)
        plt.xlabel(numeric_field)
        plt.ylabel(numeric_field2)
        plt.title(f"{numeric_field} vs {numeric_field2}")
        plt.show()

## 6. Conclusion
This notebook demonstrated loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library.

**Key findings**:
- Record sets and their fields can be identified and accessed via their Croissant `@id`s.
- Data can be filtered, normalized, and grouped for downstream analysis.
- Common EDA and visualization steps reveal data structure and basic patterns.

Use the listed field and record set `@id`s to further expand your analysis and adapt for your research workflows.